# The RBF defence — a Gaussian-kernel detector, built and attacked

**Part 1** builds the RBF (radial-basis-function) model: cluster centres initialised by KMeans, per-centre spreads (σ), and a variant restricted to the non-removal features. **Part 2** visualises what was built — the centres in t-SNE space and how similar they are to each other. **Part 3** attacks it: PGD-min, a dedicated RBF-loss attack, and GKDE, across the L1/L2/L∞ norms.

**Where this fits:** Section 03 built the attacks; this section asks how the **defences** hold up against them — above all against the σ attack that grew out of `03-attacks/cw_sine_lineage.ipynb`. The same defences, measured in the final organised protocol, are `06-thesis-evaluation/`.

*Consolidated from `RBF2.ipynb`, `RBF.ipynb`, `RBF_visualization.ipynb` and `adverserial_attack_RBF.ipynb` (originally in `mal_adv3`). The second attack notebook's long analysis bulk duplicates `03-attacks/pgd_analysis.ipynb` and is not repeated. Outputs cleared.*

## Part 1 — Building the RBF model

### Step 1 — Setup: data and the trained DNN/AT models it is compared with

In [ ]:
#RBF_models

download_links = [
                  'https://drive.google.com/uc?id=1-8lJXLdAl_4NdDwzw9kFML0aiOCTrI9f',
                  'https://drive.google.com/uc?id=1-OHACrNCt0yKBbdqQPVfNZcjKt5_jxKD',
                  'https://drive.google.com/uc?id=1-KeXJXtU1_6m9JOhormeVwigy0myX3HL',
                  'https://drive.google.com/uc?id=1-13RDdZqnrNkdHg3D8PC5KI0CZREwlsz',
                  'https://drive.google.com/uc?id=1-8LjsCdzKh6asxCFsYLiQZbSEXXKSQBP',

]

import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

In [ ]:
# Step 1: Install gdown
!pip install gdown

# Step 2: Define the Google Drive file ID and download path
file_id = '1hYbGvflzRC_V0yvCXYgqqdWlBuiMy3xC'  # Replace with your file ID
output_path = 'model.pth'

# Step 3: Download the file using gdown
import gdown

# Construct the full URL for the file
url = f'https://drive.google.com/uc?id={file_id}'

# Download the file
gdown.download(url, output_path, quiet=False)

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()
print('the proportion of malwares : ', n_mal/(n_mal+n_ben))

# Combine features and labels into datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 128 #32
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
del train_dataset, val_dataset, test_dataset, y_train, y_val, y_test

In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

### Step 2 — Initialising centres and spreads with KMeans

In [ ]:
from sklearn.cluster import KMeans

# Function to initialize centers and sigmas separately for benign and malware samples using KMeans clustering
def initialize_centers_sigmas_separate(data_loader, num_centers_per_class, removal_array = removal_array, non_removal_features = False):
    centers_benign = []
    centers_malware = []
    non_removal_mask = torch.logical_not(removal_array).to('cpu')

    # Collect data into a single tensor
    all_data = torch.cat([batch for batch, _ in data_loader], dim=0)
    if non_removal_features:
        all_data = all_data[:,non_removal_mask]

    all_data = all_data.numpy()
    # Collect labels into a single tensor
    all_labels = torch.cat([labels for _, labels in data_loader], dim=0)
    all_labels = all_labels.numpy()

    # Separate benign and malware samples
    benign_indices = np.where(all_labels == 0)[0]
    malware_indices = np.where(all_labels == 1)[0]

    # Clustering for benign samples
    benigns = all_data[benign_indices]
    if non_removal_features:
        subset_benigns = benigns
    else:
        subset_benigns = benigns[:20000]

    kmeans_benign = KMeans(n_clusters=num_centers_per_class, init='k-means++', n_init='auto')
    kmeans_benign.fit(subset_benigns)
    centers_selected_benign = kmeans_benign.cluster_centers_

    # Clustering for malware samples
    kmeans_malware = KMeans(n_clusters=num_centers_per_class, init='k-means++', n_init='auto')
    kmeans_malware.fit(all_data[malware_indices])
    centers_selected_malware = kmeans_malware.cluster_centers_

    # Combine selected centers
    all_centers = np.concatenate([centers_selected_benign, centers_selected_malware], axis=0)

    return all_centers

In [ ]:
all_centers = initialize_centers_sigmas_separate(train_loader, 500, removal_array = removal_array, non_removal_features = False)

In [ ]:
all_centers.shape

In [ ]:
# Save the array to a file
np.save('all_centers.npy', all_centers)

In [ ]:
# Replace negative values with zero
all_centers = np.maximum(all_centers, 0.)
all_centers[0][:30]

### Step 3 — The non-removal-features variant

In [ ]:
all_centers_non_removal_features = initialize_centers_sigmas_separate(train_loader, 500, removal_array = removal_array, non_removal_features = True)

In [ ]:
print(all_centers_non_removal_features.shape)

In [ ]:
all_centers_non_removal_features[0][:20]

In [ ]:
# Save the array to a file
np.save('all_centers_non_removal_features.npy', all_centers_non_removal_features)

In [ ]:
# Load the array from the file
all_centers = np.load('all_centers.npy')

In [ ]:
# Move all_centers back to GPU
all_centers = torch.tensor(all_centers, device=device)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from scipy.spatial.distance import cdist

# Step 2: Determine spread (σ)
d_max = np.max(cdist(all_centers.cpu(), all_centers.cpu(), 'euclidean'))
sigma = d_max / np.sqrt(2 * len(all_centers))

In [ ]:
sigma

In [ ]:
all_centers_non_removal_features = torch.tensor(all_centers_non_removal_features, device=device)

### Step 4 — Training and evaluation

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    #os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from sklearn.cluster import KMeans

# Define the RBF model
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output

# Function to evaluate the model on a dataset
def evaluate_model(model, data_loader, criterion, removal_array = removal_array, non_removal_features = False):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    non_removal_mask = torch.logical_not(removal_array).to('cpu')

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            if non_removal_features:
                batch_x, batch_y = batch_x[:,non_removal_mask].to(torch.float32).to(device), batch_y.to(device)  # Move data to GPU
            else:
                batch_x, batch_y = batch_x.to(torch.float32).to(device), batch_y.to(device)  # Move data to GPU
            output = model(batch_x)
            loss = criterion(output, batch_y.squeeze().to(torch.long))
            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct_predictions += (predicted == batch_y.squeeze()).sum().item()

    avg_loss = total_loss / len(data_loader)
    accuracy = correct_predictions / len(data_loader.dataset)

    return avg_loss, accuracy

In [ ]:
def model_evaluation(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for X_batch, labels_batch in test_loader:
            X_batch, labels_batch = X_batch.to(torch.float32).to(device), labels_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.topk(outputs, k=1)
            predictions.extend(predicted.squeeze().tolist())
            true_labels.extend(labels_batch.squeeze().tolist())

    # Convert predictions and true labels to numpy arrays
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate and print test accuracy
    accuracy = accuracy_score(true_labels, predictions)
    balanced_acc = balanced_accuracy_score(true_labels, predictions)
    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Test balanced Accuracy: {balanced_acc:.4f}')

    # Calculate and print precision, recall, and F1-score
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)


    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

        # Calculate and print true positives, true negatives, false positives, and false negatives
    TP = ((predictions == 1) & (true_labels == 1)).sum()
    TN = ((predictions == 0) & (true_labels == 0)).sum()
    FP = ((predictions == 1) & (true_labels == 0)).sum()
    FN = ((predictions == 0) & (true_labels == 1)).sum()

    print(f'True Positives (TP): {TP}')
    print(f'True Negatives (TN): {TN}')
    print(f'False Positives (FP): {FP}')
    print(f'False Negatives (FN): {FN}')

    # Calculate and print False Negative Rate (FNR) and False Positive Rate (FPR)
    FNR = (FN / (FN + TP)) * 100
    FPR = (FP / (FP + TN)) * 100

    print(f'False Negative Rate (FNR): {FNR:.4f}')
    print(f'False Positive Rate (FPR): {FPR:.4f}')

    return balanced_acc

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model = RBFModel(1000, 2, all_centers, [sigma], kernel)
model = model.to(device)

# Load the model state dictionary
model.load_state_dict(torch.load(output_path))

In [ ]:
# Loss function
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class
criterion = nn.CrossEntropyLoss(weight=class_weights)
#criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)  # Decrease lr by a factor of 0.95 every 1 epoch

# Training loop
num_epochs = 50
best_val_accuracy = 0.0
best_model_state_dict = None

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    correct_predictions = 0

    # Print the current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print('************************************')
    print(f'Current Learning Rate: {current_lr}')

    for i, (batch_x, batch_y) in enumerate(train_loader):
        # Remove features based on removal_array
        if non_removal_features:
            batch_x = batch_x[:, non_removal_mask].to(torch.float32).to(device)  # Select features
        else:
            batch_x = batch_x.to(torch.float32).to(device)  # Move all features to GPU
        batch_y = batch_y.to(device)  # Move labels to GPU

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y.squeeze().to(torch.long))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct_predictions += (predicted == batch_y.squeeze()).sum().item()

        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item()}')
            print('sigma : ',model.sigmas.detach())

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss}, Train Accuracy: {accuracy}')

    # Step the scheduler
    scheduler.step()



    # Evaluate on validation set
    val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, removal_array = removal_array, non_removal_features=non_removal_features)
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')
    balanced_acc = model_evaluation(model, val_loader, device)

    # Check if the current model has the best validation accuracy
    if balanced_acc > best_val_accuracy:
        best_val_accuracy = balanced_acc
        print(f"New best validation accuracy: {best_val_accuracy}")
        # Save the current best model state dict
        best_model_state_dict = model.state_dict()

# Save the best model
if best_model_state_dict is not None:
    torch.save(best_model_state_dict, 'best_model_laplacian.pth')
    print("Best model saved with validation accuracy:", best_val_accuracy)
else:
    print("No improvement in validation accuracy.")

In [ ]:
model_evaluation(model, test_loader, device)

In [ ]:
best_model_state_dict = model.state_dict()
torch.save(best_model_state_dict, 'best_model_gaussian.pth')

In [ ]:
# Step 2: Determine spread (σ)
d_max = np.max(cdist(all_centers_non_removal_features.cpu(), all_centers_non_removal_features.cpu(), 'euclidean'))
sigma = d_max / np.sqrt(2 * len(all_centers_non_removal_features))
sigma

In [ ]:
non_removal_mask = torch.logical_not(removal_array).to('cpu')

def model_evaluation(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for X_batch, labels_batch in test_loader:
            X_batch, labels_batch = X_batch[:,non_removal_mask].to(torch.float32).to(device), labels_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.topk(outputs, k=1)
            predictions.extend(predicted.squeeze().tolist())
            true_labels.extend(labels_batch.squeeze().tolist())

    # Convert predictions and true labels to numpy arrays
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate and print test accuracy
    accuracy = accuracy_score(true_labels, predictions)
    balanced_acc = balanced_accuracy_score(true_labels, predictions)
    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Test balanced Accuracy: {balanced_acc:.4f}')

    # Calculate and print precision, recall, and F1-score
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)


    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

        # Calculate and print true positives, true negatives, false positives, and false negatives
    TP = ((predictions == 1) & (true_labels == 1)).sum()
    TN = ((predictions == 0) & (true_labels == 0)).sum()
    FP = ((predictions == 1) & (true_labels == 0)).sum()
    FN = ((predictions == 0) & (true_labels == 1)).sum()

    print(f'True Positives (TP): {TP}')
    print(f'True Negatives (TN): {TN}')
    print(f'False Positives (FP): {FP}')
    print(f'False Negatives (FN): {FN}')

    # Calculate and print False Negative Rate (FNR) and False Positive Rate (FPR)
    FNR = (FN / (FN + TP)) * 100
    FPR = (FP / (FP + TN)) * 100

    print(f'False Negative Rate (FNR): {FNR:.4f}')
    print(f'False Positive Rate (FPR): {FPR:.4f}')

    return balanced_acc

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 2.5
kernel = 'gaussian'

model_non_removal = RBFModel(1000, 2, all_centers_non_removal_features, [sigma], kernel)
model_non_removal = model_non_removal.to(device)

# Loss function
#criterion = nn.CrossEntropyLoss()
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(model_non_removal.parameters(), lr=0.002)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)  # Decrease lr by a factor of 0.95 every 1 epoch

# Training loop
num_epochs = 100
best_val_accuracy = 0.0
best_model_state_dict = None

for epoch in range(num_epochs):
    model_non_removal.train()
    total_loss = 0.0
    correct_predictions = 0

    # Print the current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print('************************************')
    print(f'Current Learning Rate: {current_lr}')

    for i, (batch_x, batch_y) in enumerate(train_loader):
        # Remove features based on removal_array
        if non_removal_features:
            batch_x = batch_x[:, non_removal_mask].to(torch.float32).to(device)  # Select features
        else:
            batch_x = batch_x.to(torch.float32).to(device)  # Move all features to GPU
        batch_y = batch_y.to(device)  # Move labels to GPU

        optimizer.zero_grad()
        output = model_non_removal(batch_x)
        loss = criterion(output, batch_y.squeeze().to(torch.long))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct_predictions += (predicted == batch_y.squeeze()).sum().item()

        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item()}')
            print('sigma : ',model_non_removal.sigmas.detach())

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss}, Train Accuracy: {accuracy}')

    # Step the scheduler
    scheduler.step()

    # Evaluate on validation set
    val_loss, val_accuracy = evaluate_model(model_non_removal, val_loader, criterion, removal_array = removal_array, non_removal_features=non_removal_features)
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')
    balanced_acc = model_evaluation(model_non_removal, test_loader, device)

    # Check if the current model has the best validation accuracy
    if balanced_acc > best_val_accuracy:
        best_val_accuracy = balanced_acc
        print(f"New best validation accuracy: {best_val_accuracy}")
        # Save the current best model state dict
        best_model_state_dict = model_non_removal.state_dict()

# Save the best model
if best_model_state_dict is not None:
    torch.save(best_model_state_dict, 'best_model_laplacian.pth')
    print("Best model saved with validation accuracy:", best_val_accuracy)
else:
    print("No improvement in validation accuracy.")

In [ ]:
model.sigmas.detach()

In [ ]:
# After training, you can extract the trained parameters
trained_centers = model.centers.detach()
trained_sigmas = model.sigmas.detach()

# Print or use the trained parameters
print("Trained Centers:", trained_centers)
print("Trained Sigmas:", trained_sigmas)

In [ ]:
model_evaluation(model_non_removal_features, test_loader, device)

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')

model = RBFModel(1000, 2, all_centers_non_removal_features, [sigma_non_removal_features])
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)  # Decrease lr by a factor of 0.95 every 1 epoch

# Training loop
num_epochs = 30
best_val_accuracy = 0.0
best_model_state_dict = None

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    correct_predictions = 0

    # Print the current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print('************************************')
    print(f'Current Learning Rate: {current_lr}')

    for i, (batch_x, batch_y) in enumerate(train_loader):
        # Remove features based on removal_array
        if non_removal_features:
            batch_x = batch_x[:, non_removal_mask].to(torch.float32).to(device)  # Select features
        else:
            batch_x = batch_x.to(torch.float32).to(device)  # Move all features to GPU
        batch_y = batch_y.to(device)  # Move labels to GPU

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y.squeeze().to(torch.long))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct_predictions += (predicted == batch_y.squeeze()).sum().item()

        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item()}')

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss}, Train Accuracy: {accuracy}')

    # Step the scheduler
    scheduler.step()

    # Evaluate on validation set
    val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, removal_array = removal_array, non_removal_features=non_removal_features)
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')

    # Check if the current model has the best validation accuracy
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        print(f"New best validation accuracy: {best_val_accuracy}")
        # Save the current best model state dict
        best_model_state_dict = model.state_dict()

# Save the best model
if best_model_state_dict is not None:
    torch.save(best_model_state_dict, 'best_model3.pth')
    print("Best model saved with validation accuracy:", best_val_accuracy)
else:
    print("No improvement in validation accuracy.")

In [ ]:
# After training, you can extract the trained parameters
trained_centers = model.centers.detach().cpu().numpy()
trained_sigmas = model.sigmas.detach().cpu().numpy()

# Print or use the trained parameters
print("Trained Centers:", trained_centers)
print("Trained Sigmas:", trained_sigmas)

In [ ]:
trained_centers.shape

In [ ]:
trained_centers[0][:100]

In [ ]:
# Re-instantiate the model
all_centers = torch.rand((1000, 10000))
sigma = 5.
model = RBFModel(1000, 2, all_centers, [sigma])

# Load the saved model state dictionary
model_state_dict = torch.load('/content/drive/MyDrive/best_model.pth')

# Load the model state dictionary into the model
model.load_state_dict(model_state_dict)

model = model.to(device)

# Ensure the model is in evaluation mode
model.eval()

In [ ]:
# Re-instantiate the model
model_non_removal_features = RBFModel(1000, 2, all_centers_non_removal_features, [sigma_non_removal_features])

# Load the saved model state dictionary
model_state_dict = torch.load('best_model_non_removal_features3.pth')

# Load the model state dictionary into the model
model_non_removal_features.load_state_dict(model_state_dict)

model_non_removal_features = model_non_removal_features.to(device)

# Ensure the model is in evaluation mode
model_non_removal_features.eval()

In [ ]:
def model_evaluation(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for X_batch, labels_batch in test_loader:
            X_batch, labels_batch = X_batch[:, non_removal_mask].to(torch.float32).to(device), labels_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.topk(outputs, k=1)
            predictions.extend(predicted.tolist())
            true_labels.extend(labels_batch.tolist())

    # Convert predictions and true labels to numpy arrays
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate and print test accuracy
    accuracy = accuracy_score(true_labels, predictions)
    balanced_acc = balanced_accuracy_score(true_labels, predictions)
    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Test balanced Accuracy: {balanced_acc:.4f}')

    # Calculate and print precision, recall, and F1-score
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)

    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

    # Calculate and print true positives, true negatives, false positives, and false negatives
    TP = ((predictions == 1) & (true_labels == 1)).sum()
    TN = ((predictions == 0) & (true_labels == 0)).sum()
    FP = ((predictions == 1) & (true_labels == 0)).sum()
    FN = ((predictions == 0) & (true_labels == 1)).sum()

    print(f'True Positives (TP): {TP}')
    print(f'True Negatives (TN): {TN}')
    print(f'False Positives (FP): {FP}')
    print(f'False Negatives (FN): {FN}')

    # Calculate and print False Negative Rate (FNR) and False Positive Rate (FPR)
    FNR = (FN / (FN + TP)) * 100
    FPR = (FP / (FP + TN)) * 100

    print(f'False Negative Rate (FNR): {FNR:.4f}')
    print(f'False Positive Rate (FPR): {FPR:.4f}')

## Part 2 — Visualising the centres

### Step 5 — The centres in t-SNE space

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
#all_centers = torch.rand((1000, 10000))
model = RBFModel(1000, 2, all_centers, [sigma], kernel)
model = model.to(device)

# Load the model state dictionary
model.load_state_dict(torch.load('/content/model.pth', map_location=torch.device(device)))

In [ ]:
# Step 1: Initialize centers and sigmas
all_centers2 = model.centers.detach()
all_centers2.shape

In [ ]:
all_data = torch.cat([batch for batch, _ in train_loader], dim=0)
labels = torch.cat([label.squeeze() for _, label in train_loader], dim=0)
print(all_data.shape)
print(labels.shape)

In [ ]:
from sklearn.manifold import TSNE

# Assuming X contains your features and y contains your labels
# X.shape = (number of samples, number of features)
# y.shape = (number of samples,)
tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(all_data[:5000])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot with Seaborn
sns.scatterplot(x=X_embedded[:5000, 0], y=X_embedded[:5000, 1], hue=labels[:5000], palette=['b', 'r'], alpha=0.7)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE Visualization of Malware vs Benign Samples')
plt.legend(title='Label', loc='upper right', labels=['Benign', 'Malware'])
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Assuming X is your original high-dimensional data
# Assuming labels is your array of labels
# Assuming centers is a numpy array of shape (1000, original_dim) representing the 1000 high-dimensional centers

# Combine original data and centers for t-SNE transformation
combined_data = np.vstack([all_data[:10000], all_centers,all_centers2])

# Apply t-SNE to the combined data
tsne = TSNE(n_components=2, random_state=42)
X_embedded_combined = tsne.fit_transform(combined_data)

In [ ]:
# Extract the embedded data and centers
X_embedded = X_embedded_combined[:10000]  # Original data points
centers_embedded = X_embedded_combined[10000:10500]  # Centers
centers_embedded2 = X_embedded_combined[10500:11000]  # Centers
centers_embedded3 = X_embedded_combined[11000:11500]  # Centers
centers_embedded4 = X_embedded_combined[11500:12000]  # Centers

In [ ]:
# Create a larger plot
plt.figure(figsize=(12, 8))

# Plot with Seaborn
# First, plot the data points
#scatter = sns.scatterplot(x=X_embedded[:, 0], y=X_embedded[:, 1], hue=labels[:10000], palette=['b', 'r'], alpha=0.7, marker='o')

# Then, plot the first set of centers
plt.scatter(centers_embedded[:, 0], centers_embedded[:, 1], alpha=0.5, marker='x', s=100, c='black', label='Centers 1')

# Plot the second set of centers
plt.scatter(centers_embedded2[:, 0], centers_embedded2[:, 1], alpha=0.5, marker='x', s=100, c='green', label='Centers 2')

# Then, plot the first set of centers
plt.scatter(centers_embedded3[:, 0], centers_embedded3[:, 1], alpha=0.7, marker='*', s=50, c='cyan', label='Centers 3')

# Plot the second set of centers
plt.scatter(centers_embedded4[:, 0], centers_embedded4[:, 1], alpha=0.7, marker='*', s=50, c='pink', label='Centers 4')

# Customize legend
plt.legend(title='Legend', loc='upper right')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE Visualization of Data and Centers')
plt.show()

### Step 6 — What the Gaussian layer outputs, per class

In [ ]:
for x,y in test_loader:
  benigns = x[y.squeeze()== 0]
  malwares = x[y.squeeze()== 1]
  print(benigns.shape)
  print(malwares.shape)
  break

In [ ]:
radial_out = model.gaussian(benigns, model.centers, model.sigmas)

In [ ]:
radial_out.shape

In [ ]:
radial_out[0]

In [ ]:
radial_out2 = model.gaussian(malwares, model.centers, model.sigmas)

In [ ]:
radial_out2[0]

### Step 7 — How similar are the centres to each other?

# similarity of centers

In [ ]:
import numpy as np

# Define the vectors
vector1 = np.array([1, 0, 1, 0, 0, 0, 0])
vector2 = np.array([1, 1, 0, 0, 0, 0, 0])

# Normalize the vectors
norm_vector1 = vector1 / np.linalg.norm(vector1)
norm_vector2 = vector2 / np.linalg.norm(vector2)

print(norm_vector1)
print(norm_vector2)

In [ ]:
# Calculate the dot product of the normalized vectors
cosine_similarity = np.dot(norm_vector1, norm_vector2)

# Print the result
print(f"Cosine Similarity: {cosine_similarity}")

In [ ]:
model.centers.shape

In [ ]:
import torch.nn.functional as F

# Assuming centers is a tensor of shape [1000, 10000]
centers = model.centers

# Normalize the centers
centers_normalized = F.normalize(centers, p=2, dim=1)

# Compute the pairwise cosine similarity
pairwise_similarity = torch.matmul(centers_normalized, centers_normalized.T)

print("Pairwise similarity matrix shape:", pairwise_similarity.shape)

In [ ]:
pairwise_similarity

In [ ]:
(pairwise_similarity>0.7).sum()

In [ ]:
(pairwise_similarity<-0.7).sum()

In [ ]:
results = (pairwise_similarity[0]>0.7)

# Get the indices where results is True
indices = torch.nonzero(results).squeeze()

print("Indices where results is True:", indices.tolist())

In [ ]:
non_unique = set()
unique = []
for i in range(1000):

  if i not in non_unique:
    results = (pairwise_similarity[i] > 0.95)
    indices = torch.nonzero(results)

    for element in indices:
        non_unique.add(element.item())

    unique.append(i)

len(unique)

## Part 3 — Attacking with and against the RBF model

### Step 8 — Setup and the PGD-min baseline

In [ ]:
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model_gaussian_1000 = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000 = model_gaussian_1000.to(device)

# Load the model state dictionary
model_gaussian_1000.load_state_dict(torch.load('/content/best_model_gaussian_1000.pth', map_location=torch.device(device)))

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 1144))
model_gaussian_1000_nonremoval = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000_nonremoval = model_gaussian_1000_nonremoval.to(device)

# Load the model state dictionary
model_gaussian_1000_nonremoval.load_state_dict(torch.load('/content/best_model_gaussian_1000_nonremoval.pth', map_location=torch.device(device)))

In [ ]:
for x,y in test_loader:
  print(x.shape)
  print(y.shape)
  break

mals = x[y.squeeze()==1]
mals_y = y[y.squeeze()==1]
mals.shape

In [ ]:
def pgd_min(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        #print('*********************************************************')
        #print('t : ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print('loss : ',loss)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
            #print('torch.abs(perturbation).sum(dim=-1) : ',torch.abs(perturbation).sum(dim=-1))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            #print('l2norm ; ',l2norm)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x_next - x_var) <= 1e-6
            gradients = gradients * un_mod

            val, _ = torch.topk(torch.abs(gradients), 1)
            #print('max(grad) : ',val)
            perturbation = (torch.abs(gradients) >= val.expand_as(gradients)).float() * torch.sign(gradients).float()
            # stop perturbing the examples that are successful to evade the victim
            outputs = model(x_next)
            _, predicted = torch.topk(outputs, k=1)
            done = (predicted != y).squeeze()
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
adv = pgd(mals.to(torch.float32), mals_y, model_AT_rFGSM, insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
model_AT_rFGSM(adv)

In [ ]:
adv2 = pgd_min(mals.to(torch.float32), mals_y, model_AT_rFGSM, insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
model_AT_rFGSM(adv2)

### Step 9 — The RBF-loss attack

In [ ]:
def rbf(x, y, model, RBFModel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',
        initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.

    :param x: Feature vector
    :param y: Ground truth labels
    :param model: Neural network model
    :param RBFModel: Gaussian model for KDE
    :param insertion_array: Array for insertion operations
    :param removal_array: Array for removal operations
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf', 'l2', 'l1')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    #target's class
    traget_labels = torch.zeros_like(y.view(-1).long())

    # Compute natural loss and penalty_factor
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), traget_labels)
    kde = criterion(RBFModel(x), traget_labels)
    penalty_factor = (loss_natural / kde).detach()

    # Initialize starting point
    x_next = get_x0(x.clone(), initial_rounding_threshold, is_sample)

    for t in range(k):
        x_var = x_next.clone().detach().requires_grad_(True)
        decayed_penalty_factor = penalty_factor * (1 - t / k)

        # Compute loss
        loss = get_loss_rbf(x_var, y, model, RBFModel, decayed_penalty_factor)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)

        pos_insertion = (x_var <= 0.5).float()
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5).float() * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion

        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
        elif norm == 'l2':
            l2norm = torch.norm(gradients, dim=-1, keepdim=True)
            perturbation = gradients / (l2norm + 1e-20)
        elif norm == 'l1':
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.
        else:
            raise ValueError("Expect 'l1', 'l2', or 'linf' norm.")

        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    if random:
        round_threshold = torch.rand_like(x_next)
    x_next = round_x(x_next, round_threshold=round_threshold)

    outputs = model(x_next)
    loss_adv = criterion(outputs, traget_labels).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size(0) * 100:.3f}%.")

    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
def rbf(x, y, model, RBFModel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))
    kde = criterion(RBFModel(x), torch.zeros_like(y.view(-1).long()))
    penalty_factor = (loss_natural/kde).detach()

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Multi-step PGD
    for t in range(k):
        #print('*************************************************')
        #print('t : ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        #y_model = model(x_var)

        decayed_penalty_factor = penalty_factor * (1 - t / k)
        #print('t : ',t)
        #print('decayed_penalty_factor : ',decayed_penalty_factor)


        print('decayed_penalty_factor : ',decayed_penalty_factor)

        # Compute loss
        loss = get_loss_rbf(x_var,y,model, RBFModel, decayed_penalty_factor)
        #print('loss : ',loss)
        #print('loss_natural : ' ,criterion(model(x_var), y.view(-1).long()))
        #loss,_ = get_loss_kde(x_var,y,model,bens, bandwidth, penalty_factor)
        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))
        #pos_insertion = (x_var <= 0.5) * 1 * insertion_array
        pos_insertion = (x_var <= 0.5) * 1
        grad4insertion = (gradients >= 0) * pos_insertion * gradients

        pos_removal = (x_var > 0.5) * 1 * removal_array
        grad4removal = (gradients < 0) * pos_removal * gradients

        gradients = grad4removal + grad4insertion
        #gradients =  grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()
            #print('torch.abs(perturbation).sum(dim=-1) : ',torch.abs(perturbation).sum(dim=-1))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            #print('l2norm ; ',l2norm)
            perturbation = (gradients / l2norm).float()
            perturbation = torch.minimum(torch.tensor(1., dtype=x.dtype, device=x.device), gradients / l2norm).float()
            perturbation = torch.maximum(torch.tensor(-1., dtype=x.dtype, device=x.device), perturbation).float()
            perturbation[torch.isnan(perturbation)] = 0.
            perturbation[torch.isinf(perturbation)] = 1.

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            #print('un_mod',un_mod.sum(dim=-1))
            gradients = gradients * un_mod

            val, _ = torch.topk(torch.abs(gradients), 1)
            #print('max_grad', val)
            perturbation = (torch.abs(gradients) >= val.expand_as(gradients)).float() * torch.sign(gradients).float()
            # stop perturbing the examples that are successful to evade the victim
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Feasible projection
    #x_next = or_float_tensors(x_next, x)

    # Compute adversarial loss
    outputs = model(x_next)
    loss_adv = criterion(outputs, torch.zeros_like(y.view(-1).long())).data
    done = get_done(x_next, y, model)

    if is_report_loss_diff:
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")

    # Replace with natural if adversarial loss is higher
    #print('loss_natural : ',loss_natural)
    #print('loss_adv : ',loss_adv)
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    return x_next

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

In [ ]:
def get_loss_rbf(adv_x: torch.Tensor, y: torch.Tensor, model: nn.Module, RBFModel: nn.Module, penalty_factor: torch.Tensor) -> torch.Tensor:

    target_labels = torch.zeros_like(y.view(-1).long())
    criterion = nn.CrossEntropyLoss()

    # Compute CE loss
    outputs = model(adv_x)
    ce = criterion(outputs, target_labels)

    # Compute KDE loss
    outputs_rbf = RBFModel(adv_x)
    kde = criterion(outputs_rbf, target_labels)

    # Combine the losses with the penalty factor
    loss = ce + penalty_factor * kde

    return loss

In [ ]:
adv4 = rbf(mals.to(torch.float32), mals_y, model_AT_rFGSM, model_gaussian_1000, insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
model_AT_rFGSM(adv4)

### Step 10 — Comparisons: PGD vs GKDE, by norm (L1)

#comparing two approaches:

 1: loss based on its class and maximize the loss

 2: loss based on the goal's class and minimize the loss





# `l1 attack`

In [ ]:
adv2 = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 0, insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
model_AT_rFGSM(mals.to(torch.float32))

In [ ]:
model_gaussian_1000(adv)

In [ ]:
model_gaussian_1000_nonremoval(adv[:, non_removal_mask])

In [ ]:
model_gaussian_1000_nonremoval(adv2[:, non_removal_mask])

In [ ]:
model_gaussian_1000(adv2)

### Step 11 — Comparisons: L∞

# `linf attack`

In [ ]:
adv = pgd(mals.to(torch.float32), mals_y, model_AT_rFGSM, insertion_array, removal_array, k=100, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
adv2 = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 0, insertion_array, removal_array, k=100, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

### Step 12 — Comparisons: the two loss goals (L1, L2, L∞)

#comparing two attacks:
loss based on the goal's class and minimize the loss

 1: without loss of rbf

 2: with loss of rbf

# `l1 attack`

In [ ]:
adv = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 0, insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
adv2 = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 10., insertion_array, removal_array, k=50, step_length=1., norm='l1', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

# `l2 attack`

In [ ]:
adv = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 0, insertion_array, removal_array, k=200, step_length=0.05, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
adv2 = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 10., insertion_array, removal_array, k=200, step_length=0.05, norm='l2', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

# `linf attack`

In [ ]:
adv = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 0, insertion_array, removal_array, k=200, step_length=0.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)

In [ ]:
adv2 = gkde(mals.to(torch.float32), mals_y, model_AT_rFGSM, 10., insertion_array, removal_array, k=200, step_length=0.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)